# ODIR-5K Model Evaluation
Comprehensive evaluation of the trained model on validation set

In [ ]:
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    classification_report, 
    confusion_matrix, 
    roc_auc_score,
    average_precision_score,
    hamming_loss,
    accuracy_score
)
import json
from pathlib import Path
from tqdm import tqdm
import sys
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Add parent directory to path
parent_dir = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(parent_dir))

from src.train import ODIRDataset, MultiLabelClassifier

# Disease classes
DISEASE_CLASSES = [
    'Normal', 'Diabetes', 'Glaucoma', 'Cataract',
    'AMD', 'Hypertension', 'Myopia', 'Other'
]

# Setup
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Device: {device}")

ModuleNotFoundError: No module named 'torch'

## Load Model and Data

In [ ]:
# Paths (relative to project root)
model_path = parent_dir / 'models' / 'best_model.pth'
val_images_path = parent_dir / 'preprocessed_data_enhanced' / 'val_images.npy'
val_labels_path = parent_dir / 'preprocessed_data_enhanced' / 'val_labels.npy'

# Load validation data
print("Loading validation data...")
val_dataset = ODIRDataset(str(val_images_path), str(val_labels_path))
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4)
print(f"Loaded {len(val_dataset)} validation samples")

# Load model
print(f"\nLoading model from {model_path}")
model = MultiLabelClassifier(num_classes=8)
checkpoint = torch.load(str(model_path), map_location=device)

if 'model_state_dict' in checkpoint:
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"✓ Loaded from epoch {checkpoint.get('epoch', 'unknown')}")
    val_loss = checkpoint.get('val_loss', None)
    if val_loss is not None:
        print(f"  Validation loss: {val_loss:.4f}")
else:
    model.load_state_dict(checkpoint)

model.to(device)
model.eval()
print("✓ Model ready")

## Run Inference

In [ ]:
print("Running inference...")

all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():
    for images, labels in tqdm(val_loader, desc="Evaluating"):
        images = images.to(device)
        labels = labels.to(device)
        
        outputs = model(images)
        probs = torch.sigmoid(outputs)
        preds = (probs > 0.5).float()
        
        all_preds.append(preds.cpu().numpy())
        all_labels.append(labels.cpu().numpy())
        all_probs.append(probs.cpu().numpy())

# Concatenate all batches
y_pred = np.vstack(all_preds)
y_true = np.vstack(all_labels)
y_probs = np.vstack(all_probs)

print(f"✓ Predictions shape: {y_pred.shape}")
print(f"✓ Ground truth shape: {y_true.shape}")

## Overall Metrics

In [ ]:
# Calculate overall metrics
exact_match_acc = accuracy_score(y_true, y_pred)
hamming = hamming_loss(y_true, y_pred)
per_sample_acc = (y_true == y_pred).mean(axis=1)
mean_sample_acc = per_sample_acc.mean()

macro_auc = roc_auc_score(y_true, y_probs, average='macro')
micro_auc = roc_auc_score(y_true, y_probs, average='micro')
weighted_auc = roc_auc_score(y_true, y_probs, average='weighted')

macro_ap = average_precision_score(y_true, y_probs, average='macro')
micro_ap = average_precision_score(y_true, y_probs, average='micro')

print("="*70)
print("OVERALL METRICS")
print("="*70)
print(f"Exact Match Accuracy:    {exact_match_acc:.4f} ({exact_match_acc*100:.2f}%)")
print(f"Mean Sample Accuracy:    {mean_sample_acc:.4f} ({mean_sample_acc*100:.2f}%)")
print(f"Hamming Loss:            {hamming:.4f}")
print(f"\nMacro AUC:               {macro_auc:.4f}")
print(f"Micro AUC:               {micro_auc:.4f}")
print(f"Weighted AUC:            {weighted_auc:.4f}")
print(f"\nMacro AP:                {macro_ap:.4f}")
print(f"Micro AP:                {micro_ap:.4f}")

## Per-Class Performance

In [ ]:
# Calculate per-class metrics
per_class_metrics = []

for i, disease in enumerate(DISEASE_CLASSES):
    y_true_class = y_true[:, i]
    y_pred_class = y_pred[:, i]
    y_probs_class = y_probs[:, i]
    
    # Basic counts
    tp = ((y_true_class == 1) & (y_pred_class == 1)).sum()
    tn = ((y_true_class == 0) & (y_pred_class == 0)).sum()
    fp = ((y_true_class == 0) & (y_pred_class == 1)).sum()
    fn = ((y_true_class == 1) & (y_pred_class == 0)).sum()
    
    # Metrics
    total = len(y_true_class)
    support = y_true_class.sum()
    accuracy = (tp + tn) / total
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    
    # AUC
    try:
        if len(np.unique(y_true_class)) > 1:
            auc = roc_auc_score(y_true_class, y_probs_class)
        else:
            auc = np.nan
    except:
        auc = np.nan
    
    per_class_metrics.append({
        'Disease': disease,
        'Support': int(support),
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1': f1,
        'Specificity': specificity,
        'AUC': auc,
        'TP': int(tp),
        'TN': int(tn),
        'FP': int(fp),
        'FN': int(fn)
    })

# Create DataFrame
df_metrics = pd.DataFrame(per_class_metrics)
df_metrics = df_metrics.sort_values('Support', ascending=False)

# Display table
print("\n" + "="*70)
print("PER-CLASS PERFORMANCE")
print("="*70)
display(df_metrics[['Disease', 'Support', 'Accuracy', 'Precision', 'Recall', 'F1', 'Specificity', 'AUC']].style.format({
    'Accuracy': '{:.3f}',
    'Precision': '{:.3f}',
    'Recall': '{:.3f}',
    'F1': '{:.3f}',
    'Specificity': '{:.3f}',
    'AUC': '{:.3f}'
}))

## Visualizations

In [ ]:
# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

### Performance Metrics by Disease

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Recall by disease
axes[0, 0].barh(df_metrics['Disease'], df_metrics['Recall'], color='steelblue')
axes[0, 0].set_xlabel('Recall (Sensitivity)', fontsize=12)
axes[0, 0].set_title('Recall by Disease Class', fontsize=14, fontweight='bold')
axes[0, 0].set_xlim(0, 1)
axes[0, 0].grid(axis='x', alpha=0.3)

# Precision by disease
axes[0, 1].barh(df_metrics['Disease'], df_metrics['Precision'], color='coral')
axes[0, 1].set_xlabel('Precision', fontsize=12)
axes[0, 1].set_title('Precision by Disease Class', fontsize=14, fontweight='bold')
axes[0, 1].set_xlim(0, 1)
axes[0, 1].grid(axis='x', alpha=0.3)

# F1 Score by disease
axes[1, 0].barh(df_metrics['Disease'], df_metrics['F1'], color='mediumseagreen')
axes[1, 0].set_xlabel('F1 Score', fontsize=12)
axes[1, 0].set_title('F1 Score by Disease Class', fontsize=14, fontweight='bold')
axes[1, 0].set_xlim(0, 1)
axes[1, 0].grid(axis='x', alpha=0.3)

# AUC by disease
axes[1, 1].barh(df_metrics['Disease'], df_metrics['AUC'], color='mediumpurple')
axes[1, 1].set_xlabel('AUC Score', fontsize=12)
axes[1, 1].set_title('AUC by Disease Class', fontsize=14, fontweight='bold')
axes[1, 1].set_xlim(0, 1)
axes[1, 1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

### Support vs Performance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Support vs Recall
axes[0].scatter(df_metrics['Support'], df_metrics['Recall'], s=200, alpha=0.6, color='steelblue')
for idx, row in df_metrics.iterrows():
    axes[0].annotate(row['Disease'], (row['Support'], row['Recall']), 
                    xytext=(5, 5), textcoords='offset points', fontsize=9)
axes[0].set_xlabel('Support (Number of Samples)', fontsize=12)
axes[0].set_ylabel('Recall', fontsize=12)
axes[0].set_title('Class Support vs Recall', fontsize=14, fontweight='bold')
axes[0].grid(alpha=0.3)

# Support vs AUC
axes[1].scatter(df_metrics['Support'], df_metrics['AUC'], s=200, alpha=0.6, color='mediumpurple')
for idx, row in df_metrics.iterrows():
    axes[1].annotate(row['Disease'], (row['Support'], row['AUC']), 
                    xytext=(5, 5), textcoords='offset points', fontsize=9)
axes[1].set_xlabel('Support (Number of Samples)', fontsize=12)
axes[1].set_ylabel('AUC', fontsize=12)
axes[1].set_title('Class Support vs AUC', fontsize=14, fontweight='bold')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### Confusion Matrix Summary

In [ ]:
# Create confusion matrix data for each class
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, row in df_metrics.iterrows():
    # Create 2x2 confusion matrix
    cm = np.array([[row['TN'], row['FP']], 
                   [row['FN'], row['TP']]])
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Pred Neg', 'Pred Pos'],
                yticklabels=['True Neg', 'True Pos'],
                ax=axes[i], cbar=False)
    axes[i].set_title(f"{row['Disease']}\n(Support: {row['Support']})", 
                     fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

## Key Findings Summary

In [ ]:
print("="*70)
print("KEY FINDINGS")
print("="*70)

print("\n✅ STRONG PERFORMERS:")
strong = df_metrics[df_metrics['AUC'] > 0.7].sort_values('AUC', ascending=False)
for _, row in strong.iterrows():
    print(f"  • {row['Disease']}: AUC={row['AUC']:.3f}, Recall={row['Recall']:.3f}, Precision={row['Precision']:.3f}")

print("\n⚠️  WEAK PERFORMERS (Need Improvement):")
weak = df_metrics[df_metrics['Recall'] < 0.3].sort_values('Recall')
for _, row in weak.iterrows():
    print(f"  • {row['Disease']}: Recall={row['Recall']:.3f}, Support={row['Support']}")

print("\n📊 CLASS IMBALANCE IMPACT:")
print(f"  • Most common: {df_metrics.iloc[0]['Disease']} ({df_metrics.iloc[0]['Support']} samples)")
print(f"  • Least common: {df_metrics.iloc[-1]['Disease']} ({df_metrics.iloc[-1]['Support']} samples)")
print(f"  • Imbalance ratio: {df_metrics.iloc[0]['Support'] / df_metrics.iloc[-1]['Support']:.1f}x")

print("\n💡 RECOMMENDATIONS:")
print("  1. Integrate age/gender metadata to improve age-related diseases (Diabetes, AMD, Cataract)")
print("  2. Address class imbalance with better sampling or loss weighting")
print("  3. Consider ensemble methods for rare classes (Hypertension, AMD)")
print("  4. Use AUC as primary metric instead of accuracy due to imbalance")

## Save Results

In [ ]:
# Save detailed results
results = {
    'overall_metrics': {
        'exact_match_accuracy': float(exact_match_acc),
        'mean_sample_accuracy': float(mean_sample_acc),
        'hamming_loss': float(hamming),
        'macro_auc': float(macro_auc),
        'micro_auc': float(micro_auc),
        'weighted_auc': float(weighted_auc),
        'macro_ap': float(macro_ap),
        'micro_ap': float(micro_ap)
    },
    'per_class_metrics': df_metrics.to_dict('records'),
    'num_samples': len(y_true)
}

output_path = parent_dir / 'models' / 'evaluation_results.json'
with open(str(output_path), 'w') as f:
    json.dump(results, f, indent=2)

print(f"✅ Results saved to {output_path}")